In [ ]:
import nltk
nltk.download('stopwords')

In [1]:
import numpy as np
import json
import glob
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)
import os
from docx import Document
from pptx import Presentation
import requests
import PyPDF2

In [2]:
def LoadTextData(directory, text_column=None):
    assert isinstance(directory, str), "directory must be a String."

    all_text_data = []
    files = glob.glob(directory)

    for file_path in files:
        file_name, file_extension = os.path.splitext(file_path)

        # load txt file
        if file_extension == '.txt':
            try:
                # Use 'latin-1' encoding to handle a wider range of characters
                with open(file_path, 'r', encoding='latin-1') as infile:
                    data = infile.read()
                    all_text_data.append(data)
            except Exception as e:
                print(f"An error occurred while processing {file_path}: {e}")

        # Load csv file
        elif file_extension == '.csv':
            try:
                if text_column is None:
                    raise ValueError("For CSV files, 'text_column' must be specified.")
                with open(file_path, 'r', encoding='utf-8') as infile:
                    data = pd.read_csv(infile)
                    text_data = data.iloc[:, text_column].tolist()
                    all_text_data.extend(text_data)
            except Exception as e:
                print(f"An error occurred while processing {file_path}: {e}")

        # load json file
        elif file_extension == '.json':
            try:
                with open(file_path, 'r', encoding='utf-8-sig') as infile:
                    data = json.load(infile)
                    if isinstance(data, list):
                        for item in data:
                            if 'text' in item:
                                all_text_data.append(item['text'])
                    elif isinstance(data, dict):
                        all_text_data.append(json.dumps(data))
                    else:
                        raise ValueError("JSON file structure not supported. It should be a list of objects with a 'text' key.")
            except Exception as e:
                print(f"An error occurred while processing {file_path}: {e}")

        # Load word(.docx) file
        elif file_extension == '.docx':
            doc = Document(file_path)
            text_data = [para.text for para in doc.paragraphs]
            all_text_data.extend(text_data)

        # Load powerpoint(.pptx) file
        elif file_extension == '.pptx':
            prs = Presentation(file_path)
            text_data = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
            all_text_data.extend(text_data)

        # Load pdf file
        elif file_extension == '.pdf':
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text_data = [page.extract_text() for page in reader.pages if page.extract_text()]
                all_text_data.extend(text_data)

        else:
            raise ValueError("Unsupported file type. Supported types are 'txt', 'csv', 'json', 'docx', 'pptx', and 'pdf'.")

    return all_text_data

In [3]:
stopwords = stopwords.words("english")
print (stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [4]:
data = LoadTextData(r"Sources\\arxiv-metadata-oai-snapshot.json", text_column=0)[:10000]

# Example of accessing the first entry from each file
for entry in data:
    print(entry[0:90]) 

In [5]:
def lemmatization(texts, allowed_postags=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in texts:
        # Split the text into chunks of 1,000,000 characters or less
        chunks = [text[i:i+1000000] for i in range(0, len(text), 1000000)]
        new_text = []
        for chunk in chunks:
            doc = nlp(chunk)
            for token in doc:
                if token.pos_ in allowed_postags:
                    new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return texts_out

# Specify the text_column for CSV files
data = LoadTextData(r"C:\\Projects\\Project SB\\TextModeling\\Sources\\*", text_column=0)

# Lemmatize the text data
lemmatized_texts = lemmatization(data)

# Print the first 90 characters of the first lemmatized text
print(lemmatized_texts[0][0:90])

An error occurred while processing C:\\Projects\\Project SB\\TextModeling\\Sources\arxiv-metadata-oai-snapshot.json: Extra data: line 2 column 1 (char 1689)
An error occurred while processing C:\\Projects\\Project SB\\TextModeling\\Sources\hellowether1.csv: name 'pd' is not defined
An error occurred while processing C:\\Projects\\Project SB\\TextModeling\\Sources\list.csv: name 'pd' is not defined
page | nomy submit student no no no no


In [ ]:
import spacy

# Load English tokenizer, POS tagger, parser, NER and word vectors
nlp = spacy.load("en_core_web_sm")

# Join the list of strings into a single string
text = ' '.join(data)

# Process the text
doc = nlp(text)

# Analyze syntax
for token in doc:
    print(token.text, "|", token.lemma_)
print(text[:45])

In [ ]:
def gen_words(texts):
    final = []
    for text in texts:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])

In [ ]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=100)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0][0:20])

In [ ]:
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
# print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words  = []
words_missing_in_tfidf = []
for i in range(0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] #reinitialize to be safe. You can skip this.
    tfidf_ids = [id for id, value in tfidf[bow]]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # The words with tf-idf socre 0 will be missing

    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow

In [ ]:
id2word = corpora.Dictionary(data_words)

corpus = []
for text in data_words:
    new = id2word.doc2bow(text)
    corpus.append(new)

print (corpus[0][0:20])

word = id2word[[0][:1][0]]
print (word)

In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus[:-1],
                                           id2word=id2word,
                                           num_topics=30,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [ ]:
test_doc=corpus[-1]
vector=lda_model[test_doc]
print(vector)

def Sort(sub_list):
    sub_list.sort(key=lambda x :x[1])
    sub_list.reverse()
    return(sub_list)
new_vector=Sort(vector)
print(new_vector)

In [ ]:
lda_model.save(r"C:\Projects\Project SB\TextModeling\Model\test_model.model")

In [ ]:
new_model=gensim.models.ldamodel.LdaModel.load(r"C:\Projects\Project SB\TextModeling\Model\test_model.model")

In [ ]:
test_doc=corpus[-1]
vector=lda_model[test_doc]
print(vector)

def Sort(sub_list):
    sub_list.sort(key=lambda x :x[1])
    sub_list.reverse()
    return(sub_list)
new_vector=Sort(vector)
print(new_vector)

In [ ]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis